# The Code-Mixed Pedagogical Flow Extractor

This Python Notebook contains my implementation of [task.ipynb](task.ipynb).  


## Video Selection

**Note:** You can run the code either for the 5 example videos, or for your own video. If you want to run it for the example videos, then press the video number.

If you want to run it for your own video, then press `0` when prompted and then paste the YouTube video link to the video you want to run the pipeline all.

The videos we selected are as follows:
1. [Python in Kannada - Introduction to Coding and Python | Full Course for Beginners - #1](https://www.youtube.com/watch?v=8c74mXV2lJ0&list=PLlGueSbLhZoBRnTsGiDJeTXuQCALOTN07)
2. [Vedic Maths in Kannada: Multiplication of Any Numbers by 5: MNXY * 5 = ? :Venugopal M N](https://www.youtube.com/watch?v=9AaG24KcVIo)

In [1]:
# First uninstall any conflicting whisper packages
%pip uninstall -y whisper
%pip uninstall -y openai-whisper

# Install the correct packages
%pip install yt_dlp
%pip install openai-whisper

# IMPORTANT: After running this cell, restart the kernel (Kernel -> Restart Kernel)
# to ensure the correct whisper module is imported
print("\nRESTART THE KERNEL NOW before running the next cells!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.2/182.2 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 99.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 57.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=803980 sha256=17eacf040a89445f1dbeb3f259da2600759fa3cc38630a69a8db0e61460d4594
  Stored in directory: /root/.cache/pip/wheels/61/d2/20/09ec9bef734d126cba375b15898010b6cc28578d8afdde5869
Successfully built openai-whisper

RESTART THE KERNEL NOW before running the next cells!


In [2]:
import yt_dlp
import os

# These are the 5 example video links. 
example_videos = {
    "1": "https://www.youtube.com/watch?v=8c74mXV2lJ0",
    "2": "https://www.youtube.com/watch?v=9AaG24KcVIo",
    "3": "https://www.youtube.com/watch?v=PLACEHOLDER_LINK_3",
    "4": "https://www.youtube.com/watch?v=PLACEHOLDER_LINK_4",
    "5": "https://www.youtube.com/watch?v=PLACEHOLDER_LINK_5",
}

print("Select a video to process:")
print("[0] Enter your own custom YouTube link")
for key in example_videos.keys():
    print(f"[{key}] Run example video {key}")

choice = input("Enter your choice (0-5): ")

if choice == "0":
    video_url = input("Paste your YouTube link here: ")
elif choice in example_videos:
    video_url = example_videos[choice]
    print(f"\nSelected Example {choice}: {video_url}")
else:
    print("\nInvalid choice. Defaulting to Example 1.")
    video_url = example_videos["1"]

ydl_opts = {
    'format': 'm4a/bestaudio/best', 
    'outtmpl': 'lesson_audio.%(ext)s', # This will likely save as lesson_audio.m4a
}
if os.path.exists("lesson_audio.m4a"):
    os.remove("lesson_audio.m4a")

print(f"\nDownloading audio from: {video_url} ...")
try:
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.download([video_url])
    print("\nAudio downloaded successfully as lesson_audio.mp3!")
except Exception as e:
    print(f"\nError downloading video: {e}")

Select a video to process:
[0] Enter your own custom YouTube link
[1] Run example video 1
[2] Run example video 2
[3] Run example video 3
[4] Run example video 4
[5] Run example video 5

Selected Example 2: https://www.youtube.com/watch?v=9AaG24KcVIo

[youtube] Extracting URL: https://www.youtube.com/watch?v=9AaG24KcVIo
[youtube] 9AaG24KcVIo: Downloading webpage


[youtube] 9AaG24KcVIo: Downloading android vr player API JSON
[info] 9AaG24KcVIo: Downloading 1 format(s): 140
[download] Destination: lesson_audio.m4a
[download] 100% of    5.85MiB in 00:00:00 at 13.49MiB/s  
[FixupM4a] Correcting container of "lesson_audio.m4a"

Audio downloaded successfully as lesson_audio.mp3!


## ASR

Here we do Automatic Speech Recognition (ASR) using [OpenAI Whisper](https://openai.com/index/whisper/).  
Whisper is trained on 680,000 hours of multilingual and multitask supervised data. It shows high robustness to accents, background noise and technical languages.

We will be using it for **transcription only**, and not translation. This is because I want to keep the Indic terms and code-mixed analogies constant.  

I plan to add a transliteration layer to the transcription before the pipeline. This is solely because most of us cannot read all languages, and so if it is properly transliterated to English, we can all read it.

I use the `medium` version of the model, which is about 769M parameters and requires ~5GB VRAM, since I want it to be robust to multilinguality as well as accents. I am not using `large-v2` and `large-v3`, since these are 1.5B+ parameters, and require about 10GB of VRAM to run comfortably. Additionally, the large models are slower as well. For the usecase of this task, `medium` should do fine.  

We are running the model locally. This is because I hit rate limits sending large, 10-15 minute audio files. Also, it sort of adds reproducibility, since now anyone who clones and can run the code self-contained.

I tried running the base-model for testing, and it spewed gibberish:  
"0,2,4,6,8 nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd"

to be exact... It could not at all handle the rapid kannada-english code switching in that video, would be my supposition.

In [4]:
import whisper
import time
import os

audio_file = "lesson_audio.m4a"

if not os.path.exists(audio_file):
    print(f"Error: {audio_file} not found. Please run the download cell first.")
else:
    model = whisper.load_model("large-v3") 
    
    print(f"\nTranscribing {audio_file}... this might take a few minutes.")
    start_time = time.time()
    
    # We force the language to Kannada ("kn") to prevent auto-translation to English.
    # The other parameters prevent the model from getting stuck in a repetition loop.
    result = model.transcribe(
        audio_file,
        language="kn", 
        condition_on_previous_text=False, 
        fp16=False, 
        no_speech_threshold=0.6
    )
    
    end_time = time.time()
    print(f"\nTranscription complete in {round(end_time - start_time, 2)} seconds!")
    
    transcript223 = result["text"]
    
    # Save the transcript to a text file for the next step (LLM processing)
    with open("transcript223.txt", "w", encoding="utf-8") as f:
        f.write(transcript)
        
    print("Full transcript successfully saved to 'transcript223.txt'.")

100%|██████████████████████████████████████| 2.88G/2.88G [00:12<00:00, 255MiB/s]



Transcribing lesson_audio.m4a... this might take a few minutes.

Transcription complete in 180.62 seconds!
Full transcript successfully saved to 'transcript.txt'.
